# Milestone 5 lab, notebook 5 of 8: Too many alerts

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| **5. Too many alerts** (you are here) | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and two minutes of waiting** while Airflow runs the rest of the week.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_NIGHT, WEEK, alerts, connect, night_by_night, prepare_nights, shouted, sql,
)

prepare_nights(LAST_NIGHT)

Airflow runs 5 nights, about half a minute each ...


The bonus week ran under Airflow: 7 nights, Mon 08-24 to Sun 08-30. 1 of 6 stores can be paid.


## Step 1: The careless way

**Tomas Reyes** runs store `S03`. On Monday one refund of $9.49 in his file had the wrong sign. Two tests catch it: "a refund gives money back" and "a line adds up". The row stays wrong until the store sends a corrected file. So both tests fail on Monday night, and again on Tuesday night, and every night after.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

The careless way to alert: one message for every failed test, every night, to everyone. Here is Sunday night, done that way:

In [2]:
print("\n".join(shouted(LAST_NIGHT)))

ALERT to everyone: the test a_count_for_every_day failed with 1 bad rows
ALERT to everyone: the test believable_quantity failed with 1 bad rows
ALERT to everyone: the test every_sale_has_a_product failed with 3 bad rows
ALERT to everyone: the test file_arrived_on_time failed with 1 bad rows
ALERT to everyone: the test file_has_a_new_column failed with 1 bad rows
ALERT to everyone: the test file_is_missing_a_column failed with 1 bad rows
ALERT to everyone: the test inside_opening_hours failed with 4 bad rows
ALERT to everyone: the test line_adds_up failed with 1 bad rows
ALERT to everyone: the test normal_volume failed with 1 bad rows
ALERT to everyone: the test one_person_once failed with 1 bad rows
ALERT to everyone: the test price_of_the_day failed with 6 bad rows
ALERT to everyone: the test refund_gives_money_back failed with 1 bad rows
ALERT to everyone: the test store_is_not_ready_to_pay failed with 5 bad rows
ALERT to everyone: the test supplier_contract failed with 2 bad rows
AL

Fifteen messages on one night. Now the whole week:

In [3]:
loud = [line for night in WEEK for line in shouted(night)]
about_the_refund = [line for line in loud if "refund_gives_money_back" in line or "line_adds_up" in line]
print("Messages in the week:", len(loud))
print("Messages that name one of the two tests the refund fails:", len(about_the_refund))

Messages in the week: 70
Messages that name one of the two tests the refund fails: 14


70 messages in seven nights. Fourteen of them name the two tests Tomas's refund fails, night after night.

What do people do with 70 messages, most of them the same as yesterday's? They stop reading them. Then the one new message that matters, Ines's missing file, is lost among them.

That is **alert fatigue**: people get so many alerts that they stop reading them. Its opposite is a short list where every line needs somebody.

## Step 2: Group: one problem, one alert

People act on "store `S03`, Monday", not on a test name. So the pipeline first turns failed tests into **problems**. Each problem has a **fingerprint**: a short name that stays the same from night to night.

In [4]:
with connect() as wh:
    problems = alerts.tonight(wh, LAST_NIGHT)
pd.DataFrame([{"fingerprint": a.fingerprint, "level": a.level, "to whom": a.to_whom} for a in problems])

,fingerprint,level,to whom
0,price_of_the_day|S04 Fri 08-28,ticket,Finance
1,refused|supplier_prices_SUP-B_2026-08-25.csv,ticket,"Finance, and the buying team"
2,refused|supplier_prices_SUP-C_2026-08-28.csv,ticket,"Finance, and the buying team"
3,store_day|S02|2026-08-30,ticket,Head of retail operations
4,store_day|S03|2026-08-24,ticket,Head of retail operations
5,store_day|S04|2026-08-25,ticket,Head of retail operations
6,store_day|S05|2026-08-27,ticket,Head of retail operations
7,store_day|S06|2026-08-29,ticket,Head of retail operations
8,file_arrived_on_time|S06 Thu 08-27,message,Store operations
9,file_has_a_new_column|supplier_prices_SUP-B_2026-08-25.csv,message,The data engineer on call


Fifteen failed tests on Sunday night became these fourteen problems. The refund is one row: `store_day|S03|2026-08-24`. Its two failed tests are inside it, as its reasons.

## Step 3: Send once

A problem that was already sent is not sent again. It is counted as "still open". Here is the refund's fingerprint, night by night:

In [5]:
sql("""
    SELECT night, event, level, title
    FROM ops.alert_log
    WHERE fingerprint = 'store_day|S03|2026-08-24'
    ORDER BY night
""")

,night,event,level,title
0,2026-08-24,new,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
1,2026-08-25,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
2,2026-08-26,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
3,2026-08-27,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
4,2026-08-28,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
5,2026-08-29,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
6,2026-08-30,still open,ticket,"S03 Mon 08-24: Rows were set aside, and the day does not add up"


Sent once, on Monday night. Still open on the six nights after, and not sent again.

## Step 4: Rank, and silence

Each problem has a level: **page**, **ticket** or **message**. Only a page may wake someone. A ticket is for today. A message is sent to nobody: it waits in a **digest**, a list someone reads when they can. That is the silence. Here is the digest of the week:

In [6]:
sql("""
    SELECT first_night, to_whom, title
    FROM ops.alerts
    WHERE level = 'message'
    ORDER BY first_night, title
""")

,first_night,to_whom,title
0,2026-08-25,The data engineer on call,S01 Tue 08-25 sent sales lines twice. The extra copies were removed
1,2026-08-25,The data engineer on call,supplier_prices_SUP-B_2026-08-25.csv has a column nobody agreed
2,2026-08-26,Marketing,M-0304 is the same person as an older member
3,2026-08-28,Head of retail operations,S02 Fri 08-28 had a day far from its normal day
4,2026-08-28,The data engineer on call,S02 Fri 08-28 sent sales lines twice. The extra copies were removed
5,2026-08-28,Store operations,S06 Thu 08-27 sent its file late


Copies that were removed, a day far from normal, a late file that has since arrived, the same person signed up twice. Each is worth knowing. None needs a person tonight.

## Step 5: Close

When a problem is gone, its alert is **resolved**, and the person is told once more. Aiko's late file:

In [7]:
sql("""
    SELECT title, level, first_night, last_night, resolved_night
    FROM ops.alerts
    WHERE fingerprint = 'store_day|S06|2026-08-27'
""")

,title,level,first_night,last_night,resolved_night
0,S06 Thu 08-27: No sales file arrived,ticket,2026-08-27,2026-08-27,2026-08-28


Sent on Thursday night. Resolved on Friday night, when the file arrived and the day added up. Nobody did anything. If alerts never close, nobody trusts an open one.

## Step 6: The two ways, night by night

In [8]:
night_by_night()

,night,shout: one per failed test,sent: new tickets,"still open, not sent again","new messages, kept for the digest",resolved
0,Mon 08-24,3,1,0,0,0
1,Tue 08-25,7,2,1,2,0
2,Wed 08-26,8,0,5,1,0
3,Thu 08-27,9,2,6,0,0
4,Fri 08-28,14,3,7,3,1
5,Sat 08-29,14,1,12,0,1
6,Sun 08-30,15,1,13,0,0
7,the week,70,10,44,6,2


Read the last row.

| | Shout: every failed test, every night | Grouped, sent once, ranked, silenced |
|---|---|---|
| Alerts in the bonus week | 70 | 10 tickets |
| The refund at `S03` | 14 messages | 1 ticket |
| What people do | Stop reading | Act |

Ten tickets, each sent once, each to the person who can act. The rest were counted, kept in the digest, or closed.

## What you now know

- **Alert fatigue**: people get so many alerts that they stop reading them, and then miss the one that matters.
- The careless way, one alert per failed test per night, would have sent 70 in the bonus week.
- Four steps make it 10: **group** failed tests into problems, **send once** by fingerprint, **rank** as page, ticket or message, and **silence** messages into a digest.
- An alert is **resolved** when its problem is gone, and the person is told.

**Say it in an interview:** "One alert per failed test per night would have been 70 in a week, the same refund seven nights running. I grouped failed tests into problems with a fingerprint, sent each once, ranked them, and kept messages in a digest. That left 10 tickets, and each was closed by itself when its problem was gone."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section on alert fatigue, and [Airflow: Monitoring, Logging, Alerting & Production Failures](https://www.d8loop.com/modules/knowledge-corner/airflow/monitoring-logging-alerting-and-production-failures).

Next: [notebook 6](06_fix_one_day.ipynb). Tomas's ticket is open. Close it the right way.